# Numerical uncertainty and readable centreline results
**Run this first to review your existing completed run. No reconstruction or cache rebuild is required.** It reads saved field/centre summaries and computes CE-minus-AE centreline contrasts from saved per-eddy centrelines.

Numerical intervals and zoomed panels expose small offsets hidden by extreme member trajectories. Sections stop at the actual shallowest/deepest reconstructed levels. Means and shaded intervals do not imply continuous vertical information between the fitted depths.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
HERE = Path.cwd().resolve()
ANALYSIS = next((p for p in (HERE, *HERE.parents) if (p/'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or a subdirectory')
WORK = ANALYSIS/'esp_population_composites'
for p in (ANALYSIS, WORK):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
import seacofs_tilt_tools as tilt
import population_tools as pop
pd.set_option('display.max_columns', 60)


In [ ]:
OUTPUT_ROOT = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/esp_population_composites')
RUN_PATH = None  # e.g. OUTPUT_ROOT/'39f937013b2719bf' for the original run
run = Path(RUN_PATH) if RUN_PATH else Path((OUTPUT_ROOT/'latest_run.txt').read_text().strip())
results, depths, coordinate, config = pop.load_saved_results(run)
manifest = pd.read_csv(run/'members/members.csv')
report = run/'report_v2'
report.mkdir(exist_ok=True)
print('Source run:', run)
print('Exact depths (m):', depths)
print('Units:', config['units'])
# Original run provenance stays unchanged. Record the reporting implementation separately.
import hashlib
(report/'report_provenance.json').write_text(json.dumps(dict(
    source_run=str(run), source_config=config,
    reporting_code_sha256=hashlib.sha256(Path(pop.__file__).read_bytes()).hexdigest()), indent=2))


In [ ]:
intervals = pop.centre_interval_table(results, depths, config['units'])
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(intervals.round(6))
intervals.to_csv(report/'centre_intervals.csv', index=False)
fig = pop.plot_zoomed_centres(results, depths, config['units'])
fig.savefig(report/'zoomed_centrelines.png', dpi=180)
plt.show()


## Raw polarity contrasts within each regime
The contrast is **CE minus AE shallow-to-deep centre displacement**; its negative is the contrast in deep-to-shallow tilt. Whole-eddy draws preserve covariance for eddies appearing in multiple groups. Intervals describe raw population differences, not a causal polarity effect or a spatially matched comparison. `excludes_zero` is a pointwise diagnostic, not a multiple-testing-adjusted finding.

In [ ]:
contrasts = pop.centre_contrasts(manifest, run/'members', depths,
                                n_boot=config['n_boot'], seed=config['seed'],
                                min_members=config['min_eddies'])
contrasts['units'] = config['units']
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(contrasts.round(6))
contrasts.to_csv(report/'centre_CE_minus_AE.csv', index=False)


In [ ]:
for group, result in results.items():
    figures = pop.plot_sections_3d(result, depths, coordinate, config['units'])
    for suffix, fig in figures:
        fig.suptitle(group+' — '+suffix)
        fig.savefig(report/f'{group}_{suffix}.png', dpi=180)
        plt.show()
print('Updated report:', report)
